# FLUX.2-klein-4B 로 이미지 생성하기 (Colab + ComfyUI)

Black Forest Labs 의 최신 텍스트→이미지 모델 **FLUX.2-klein-4B** 를
Google Colab(무료 GPU)에서 **ComfyUI** 로 실행하고, **Gradio** 로 UI 까지 띄우는 실습입니다.

---

## 학습 목표
- 텍스트→이미지 생성 모델의 **전체 파이프라인** 이해
- **양자화(GGUF Q4)** 가 왜 필요한지 체감 (16GB → 2.6GB)
- **ComfyUI** 의 노드 기반 동작 원리와 **API 호출** 이해
- **Gradio** 로 나만의 이미지 생성 UI 만들기

---

## 왜 이 구성인가? (핵심 개념)

| 시도 | 결과 | 이유 |
|---|---|---|
| 풀 BF16 모델 | Colab RAM 부족 | 모델 크기 ~16GB > Colab 시스템 RAM(~13GB) |
| fp8 단일 파일 | diffusers 미지원 | `from_single_file` 메서드가 없음 |
| **GGUF Q4 + ComfyUI** | **성공** | 2.6GB 로 축소, ComfyUI 가 GGUF 네이티브 지원 |

> **양자화(Quantization)**: 모델 가중치의 숫자 정밀도를 낮춰(예: 16비트→4비트)
> 크기와 메모리를 줄이는 기술. 품질 손실은 거의 없습니다.

---

## 모델 구성 (3가지 파일이 하나의 파이프라인을 이룸)

| 역할 | 파일 | 설명 |
|---|---|---|
| **디퓨전 모델** | `flux-2-klein-4b-Q4_K_M.gguf` | 텍스트를 이해해 이미지를 그려내는 핵심 (4B, 양자화 2.6GB) |
| **텍스트 인코더** | `qwen_3_4b.safetensors` | 프롬프트를 모델이 이해하는 숫자로 변환 (Qwen3-4B) |
| **VAE** | `flux2-vae.safetensors` | 압축된 latent 를 최종 픽셀 이미지로 복원 |

> 이 세 가지가 합쳐져야 비로소 "텍스트 → 이미지" 가 완성됩니다.

---

## 실습 준비
1. 상단 메뉴 **[런타임 → 런타임 유형 변경 → T4 GPU]** 선택 후 저장
2. 아래 셀을 **1번부터 순서대로** 실행 (모델 다운로드에 시간이 좀 걸립니다 ~11GB)

## [단계 1] 런타임에 GPU 가 있는지 확인

이 실습은 GPU(그래픽카드) 없이는 실행할 수 없습니다. 현재 Colab 런타임이 GPU 를 인식하고 있는지, 인식한다면 어떤 종류(T4 등)이고 VRAM(비디오 메모리)은 얼마인지 확인합니다.

GPU가 감지되지 않으면 상단 메뉴 **[런타임 → 런타임 유형 변경 → T4 GPU]** 로 바꾼 뒤 이 셀을 다시 실행해야 합니다.

In [ ]:
# 1) GPU 가 사용 가능한지 확인
import torch

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU가 없습니다. [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행하세요.")

PyTorch 버전: 2.11.0+cu128
CUDA 사용 가능: True
GPU: Tesla T4
VRAM(GB): 15.64


## [단계 2] ComfyUI 본체와 GGUF 노드 설치

**ComfyUI** 는 노드(node) 기반의 이미지 생성 도구로, 각 기능(모델 로드, 텍스트 인코딩, 샘플링 등)을 블록처럼 연결해 파이프라인을 구성합니다. 이 실습에서는 ComfyUI 를 **웹 UI 없이 API 서버로만** 사용합니다.

여기에 더해 **ComfyUI-GGUF** 라는 커스텀 노드(플러그인)를 설치합니다. ComfyUI 기본 기능은 GGUF 포맷을 읽지 못하기 때문에, 양자화 모델을 로드하려면 이 플러그인이 필요합니다.

In [ ]:
# 2) ComfyUI 본체 + GGUF 노드(플러그인) 설치
import os

%cd /content
if not os.path.isdir("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git   # ComfyUI 본체

%cd /content/ComfyUI
!pip install -q -r requirements.txt                            # ComfyUI 의존성

# GGUF 파일을 읽기 위한 커스텀 노드 설치
%cd /content/ComfyUI/custom_nodes
if not os.path.isdir("ComfyUI-GGUF"):
    !git clone https://github.com/city96/ComfyUI-GGUF.git      # GGUF 로더 노드
%cd /content/ComfyUI
!pip install -q gguf                                           # GGUF 파싱 라이브러리

print("\n완료: ComfyUI + GGUF 노드 설치")

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 43734, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 43734 (delta 21), reused 5 (delta 5), pack-reused 43692 (from 3)
Receiving objects: 100% (43734/43734), 84.42 MiB | 21.08 MiB/s, done.
Resolving deltas: 100% (29690/29690), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.8/39.8 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 93.0 MB

## [단계 3] 모델 파일 3종 다운로드

위 개요에서 설명한 대로, FLUX.2-klein 파이프라인은 **세 가지 파일**로 구성됩니다. 각 파일을 ComfyUI 가 정해둔 표준 폴더에 다운로드합니다.

- **디퓨전 모델** (2.6GB) - GGUF Q4 양자화본. 이미지 생성의 핵심 두뇌
- **텍스트 인코더** (8GB) - Qwen3-4B. 프롬프트를 숫자로 변환
- **VAE** (330MB) - latent 를 사람이 보는 픽셀 이미지로 복원

이미 받은 파일은 건너뛰도록 되어 있어, 런타임이 끊기고 다시 실행해도 처음부터 받지 않습니다. 총 약 11GB 로 시간이 꽤 걸립니다.

In [ ]:
# 3) 파이프라인 구성 파일 3종 다운로드 → ComfyUI 표준 폴더에 배치
import os
%cd /content/ComfyUI

# ComfyUI 가 모델을 찾는 표준 경로 생성
os.makedirs("models/diffusion_models", exist_ok=True)   # 디퓨전 모델
os.makedirs("models/text_encoders",  exist_ok=True)   # 텍스트 인코더
os.makedirs("models/vae",            exist_ok=True)   # VAE

def 다운로드(url, 저장경로):
    """이미 받았으면 건너뛰고, 아니면 wget 로 받는 헬퍼"""
    if os.path.exists(저장경로) and os.path.getsize(저장경로) > 1_000_000:
        print(f"  ↳ 이미 있음: {저장경로} ({os.path.getsize(저장경로)/1e9:.2f} GB)")
        return
    print(f"  ↳ 다운로드 중...")
    !wget -q --show-progress -O "{저장경로}" "{url}"

print("[1/3] 디퓨전 모델 (GGUF Q4, 약 2.6GB)")
다운로드("https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_K_M.gguf",
        "models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf")

print("[2/3] 텍스트 인코더 (Qwen3-4B, 약 8GB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors",
        "models/text_encoders/qwen_3_4b.safetensors")

print("[3/3] VAE (약 330MB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/vae/flux2-vae.safetensors",
        "models/vae/flux2-vae.safetensors")

print("\n다운로드 완료. 배치된 파일 확인:")
!ls -lh models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf \
      models/text_encoders/qwen_3_4b.safetensors \
      models/vae/flux2-vae.safetensors

/content/ComfyUI
[1/3] 디퓨전 모델 (GGUF Q4, 약 2.6GB)
  ↳ 다운로드 중...
models/diffusion_mo 100%[===================>]   2.42G  72.3MB/s    in 33s     
[2/3] 텍스트 인코더 (Qwen3-4B, 약 8GB)
  ↳ 다운로드 중...
models/text_encoder 100%[===================>]   7.49G   113MB/s    in 74s     
[3/3] VAE (약 330MB)
  ↳ 다운로드 중...
models/vae/flux2-va 100%[===================>] 320.64M  97.8MB/s    in 3.4s    

다운로드 완료. 배치된 파일 확인:
-rw-r--r-- 1 root root 2.5G Jul 30 08:38 models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf
-rw-r--r-- 1 root root 7.5G Jul 30 08:40 models/text_encoders/qwen_3_4b.safetensors
-rw-r--r-- 1 root root 321M Jul 30 08:40 models/vae/flux2-vae.safetensors


## [단계 4] ComfyUI 서버 실행

ComfyUI 를 **백그라운드 프로세스**로 띄웁니다. 이 서버는 Colab 컨테이너 내부의 `http://127.0.0.1:8188` 에서 HTTP API 로 동작하며, 다음 단계의 Gradio UI 가 이 서버에 요청을 보내는 구조입니다.

실행 후 `http 200` 응답이 확인되면 서버가 준비된 것입니다. 이 셀이 실행 중인 동안 서버가 계속 유지됩니다.

In [ ]:
# 4) ComfyUI 서버를 백그라운드로 실행 (Gradio 가 이 서버에 접속함)
#    ※ Gradio 를 쓰므로 외부 공개 터널은 필요 없습니다.
import subprocess, os, time

%cd /content/ComfyUI

# 혹시 떠 있는 예전 서버가 있으면 정리
!pkill -f "main.py" 2>/dev/null
time.sleep(2)

# ComfyUI 백그라운드 실행 (포트 8188)
os.system("nohup python main.py --listen 0.0.0.0 --port 8188 > /content/comfyui.log 2>&1 &")

# 서버가 뜰 때까지 대기 (로컬에서 200 응답이 오면 준비 완료)
print("ComfyUI 기동 대기 중...")
for _ in range(40):
    time.sleep(1)
    code = subprocess.run("curl -s -o /dev/null -w '%{http_code}' http://127.0.0.1:8188/",
                          shell=True, capture_output=True, text=True).stdout
    if code == "200":
        print("완료: ComfyUI 서버 준비 (http://127.0.0.1:8188)")
        break
else:
    print("서버가 안 뜹니다. 로그:")
    print(open("/content/comfyui.log").read()[-800:])

/content/ComfyUI
^C
ComfyUI 기동 대기 중...
완료: ComfyUI 서버 준비 (http://127.0.0.1:8188)


## [단계 5] Gradio UI 실행

**Gradio** 로 프롬프트를 입력하고 이미지를 받는 웹 UI 를 만듭니다. 이 UI 는 사용자의 입력을 받아 ComfyUI 서버의 API 로 전달하고, 생성이 끝나면 결과 이미지를 가져와 화면에 표시합니다.

실행하면 `gradio.live` 로 끝나는 **임시 공개 링크**가 출력됩니다. 이 링크를 브라우저로 열면 누구나 접속해 이미지를 생성할 수 있습니다. (링크는 이 셀이 실행 중일 때만 유효)

### 사용 순서
1. 공개 링크를 브라우저로 열기
2. 프롬프트 입력 (예: "A cat sitting on a table")
3. 해상도 / 스텝 수 / 시드 조정 (기본값 그대로도 됨)
4. **이미지 생성** 버튼 클릭 → 약 20초 대기 (첫 실행은 모델 로드로 더 걸림)

In [ ]:
# 5) Gradio UI — 프롬프트를 입력하면 이미지를 생성하는 웹 화면
#    ComfyUI 의 API 를 호출해 백엔드로 사용합니다.
import gradio as gr
import requests, time, json, urllib.parse
import numpy as np
from PIL import Image as PILImage

SERVER = "http://127.0.0.1:8188"   # Colab 안의 ComfyUI 서버 주소

# ---------- 핵심: ComfyUI API 로 이미지 생성하는 함수 ----------
def generate(prompt, negative, width, height, steps, seed):
    """프롬프트 → ComfyUI 워크플로우 JSON 조립 → API 호출 → 결과 이미지 반환"""

    # ① ComfyUI 가 이해하는 워크플로우(노드 그래프)를 딕셔너리로 표현
    #    각 노드는 역할을 가짐: 로더 → 인코더 → 샘플러 → 디코더 → 저장
    workflow = {
      "10": {"class_type": "UnetLoaderGGUF",  "inputs": {"unet_name": "flux-2-klein-4b-Q4_K_M.gguf"}},   # 디퓨전 모델 로드
      "8":  {"class_type": "CLIPLoader",      "inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "flux2"}},  # 텍스트 인코더 로드
      "6":  {"class_type": "CLIPTextEncode",  "inputs": {"text": prompt, "clip": ["8", 0]}},             # 프롬프트 인코딩
      "7":  {"class_type": "CLIPTextEncode",  "inputs": {"text": negative or "", "clip": ["8", 0]}},     # 네거티브 인코딩
      "5":  {"class_type": "EmptyLatentImage", "inputs": {"width": int(width), "height": int(height), "batch_size": 1}},  # 빈 캔버스
      "3":  {"class_type": "KSampler",        "inputs": {"seed": int(seed), "steps": int(steps), "cfg": 4.0,
                                                         "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
                                                         "model": ["10", 0], "positive": ["6", 0],
                                                         "negative": ["7", 0], "latent_image": ["5", 0]}},  # 실제 생성
      "4":  {"class_type": "VAELoader",       "inputs": {"vae_name": "flux2-vae.safetensors"}},          # VAE 로드
      "9":  {"class_type": "VAEDecode",       "inputs": {"samples": ["3", 0], "vae": ["4", 0]}},         # latent → 이미지
      "11": {"class_type": "SaveImage",       "inputs": {"images": ["9", 0], "filename_prefix": "flux2_klein"}},  # 저장
    }

    # ② 워크플로우를 ComfyUI 에 전송 (생성 요청)
    try:
        r = requests.post(f"{SERVER}/prompt", json={"prompt": workflow, "client_id": "colab"}, timeout=30)
        if r.status_code != 200:
            return None, f"요청 실패 {r.status_code}: {r.text[:300]}"
        pid = r.json()["prompt_id"]   # 작업 ID
    except Exception as e:
        return None, f"서버 연결 실패 (4번 셀 실행 확인): {e}"

    # ③ 생성이 끝날 때까지 주기적으로 결과 확인 (폴링)
    t0 = time.time()
    for _ in range(180):
        time.sleep(2)
        try:
            h = requests.get(f"{SERVER}/history/{pid}", timeout=10).json()
        except Exception:
            continue
        if pid in h:
            entry = h[pid]
            if entry.get("status", {}).get("status_str") == "error":           # ComfyUI 내부 에러
                return None, "ComfyUI 실행 에러: " + json.dumps(entry["status"].get("messages", []), ensure_ascii=False)[:500]
            outs = entry.get("outputs", {})
            if "11" not in outs or not outs["11"].get("images"):
                return None, "결과 이미지 없음"
            img = outs["11"]["images"][0]
            break
    else:
        return None, "타임아웃 (6분 초과)"

    # ④ 완성된 이미지를 다운로드해 Gradio 에 표시
    params = urllib.parse.urlencode({"filename": img["filename"], "subfolder": img.get("subfolder",""), "type": img.get("type","output")})
    rr = requests.get(f"{SERVER}/view?{params}", timeout=30)
    if rr.status_code != 200 or len(rr.content) < 1000:
        return None, f"이미지 다운로드 실패 (status={rr.status_code})"

    local = f"/content/flux2_out_{int(time.time())}.png"
    open(local, "wb").write(rr.content)
    arr = np.array(PILImage.open(local).convert("RGB"))   # numpy 배열로 변환
    return arr, f"완료 ({round(time.time()-t0,1)}초) — {int(width)}×{int(height)}, {int(steps)}스텝"


# ---------- Gradio 화면 구성 ----------
with gr.Blocks() as demo:
    gr.Markdown("# FLUX.2-klein-4B 이미지 생성기\n프롬프트를 입력하고 버튼을 누르세요. (4번 셀의 서버가 실행 중이어야 합니다)")

    with gr.Row():
        with gr.Column(scale=3):
            prompt   = gr.Textbox(label="프롬프트", value="A cow flying in the sky with wings on its back, photorealistic", lines=2)
            negative = gr.Textbox(label="네거티브 프롬프트 (선택)", value="", lines=1)
            btn      = gr.Button("이미지 생성", variant="primary")
            status   = gr.Markdown("")
        with gr.Column(scale=2):
            width  = gr.Slider(512, 1536, value=1024, step=64, label="가로 해상도")
            height = gr.Slider(512, 1536, value=1024, step=64, label="세로 해상도")
            steps  = gr.Slider(1, 8, value=4, step=1, label="스텝 수 (distilled 모델은 4 추천)")
            seed   = gr.Number(value=42, label="시드(Seed)")

    out = gr.Image(label="생성 결과")
    btn.click(generate, [prompt, negative, width, height, steps, seed], [out, status])

# 외부에서 접속 가능한 임시 공개 링크 생성
demo.launch(share=True, server_port=7860, quiet=True)

* Running on public URL: https://9b5307a2ed6adfad58.gradio.live
